# Sprint 4 — Model Research Notes (Part 1)

## Objective

Sprint 4 is a research sprint: before writing any code in `src/models/`, we
read the papers that justify our architecture and transfer-learning
decisions, extract the findings that are actually relevant to VisionLab, and
turn them into concrete design decisions for the Model Zoo.

Each reading note below follows the same structure: what the paper argues,
its experimental setup and findings, and — separately — how it applies to
VisionLab's specific situation (two fine-grained image classification
datasets, mushroom and flower, sharing one pipeline). The two are kept apart
deliberately: the paper's findings are general and citable, the VisionLab
section is our own interpretation and may age faster.

Sprint 4's reading notes are split across multiple notebooks (`part1`,
`part2`, ...) so a single file doesn't grow too long — `04_model_research_part2.ipynb`
is reserved for future notes.

## Reading Note 1: How Transferable Are Features in Deep Neural Networks?

**Source:** Yosinski, J., Clune, J., Bengio, Y., & Lipson, H. (2014).
*How Transferable Are Features in Deep Neural Networks?* NIPS 2014.
[arXiv:1411.1792](https://arxiv.org/abs/1411.1792)

### 1. What does the paper argue?

The starting observation is simple: no matter which dataset or task a CNN is
trained on, its first layer always learns the same thing — Gabor filters and
color blobs. The authors call this a **general** feature. The last layer, by
contrast, is tied directly to the specific classes — a **specific** feature.
The real question: where in the network does the general→specific
transition happen, and is it abrupt or gradual?

**Experimental setup:**

- ImageNet's 1000 classes are randomly split into two halves (A/B, 500
  classes each); two separate 8-layer networks (`baseA`, `baseB`) are
  trained from scratch, one per half.
- A layer `n` is chosen (1–7). The first `n` layers are copied from one base
  network and frozen; the remaining layers are randomly initialized and
  trained on the other dataset.
- **Selffer (BnB):** source and target are the same dataset — this is the
  control group.
- **Transfer (AnB):** source is A, target is B — this is the actual
  measurement of interest.
- Both have a "frozen" version and a fine-tuned version (marked `+`, where
  the whole network keeps learning).

### Findings

| # | Finding | Explanation |
|---|---------|-------------|
| 1 | Layers 1–2 transfer without loss | Gabor/color features really are universal |
| 2 | Performance drops in the middle layers (3–6), for **two distinct reasons** | (i) *fragile co-adaptation*: neighboring layers adapt to each other so tightly during joint training that once one layer is frozen, that co-adaptation can no longer be re-established — this is purely an optimization problem, unrelated to specificity. (ii) *representation specificity*: the features really are specific to the source task. (i) dominates in layers 3–5, (ii) dominates in layers 6–7. |
| 3 | Fine-tuning largely recovers this loss | Fine-tuning instead of freezing brings the co-adaptation loss down to nearly zero |
| 4 | Transfer + fine-tuning generalizes **better** than training from scratch | Even when the target dataset is large, having "seen" the source dataset leaves a lasting generalization boost (~1.6–2.1%) |
| 5 | Random (untrained) weights perform very poorly | Performance drops to chance level after layer 3 — the "random filters work fine" result seen in the small-network literature does not hold for deep networks |
| 6 | The more distant the source/target tasks, the weaker the transfer | But even transfer from the most distant task still beats random weights |

## 2. Evaluation for VisionLab

Our scenario (mushroom dataset + Oxford Flowers, most likely on an
ImageNet-pretrained backbone) is fairly close to this experimental setup:
two different fine-grained image classification tasks, run through the same
pipeline.

**Concrete decisions this supports:**

- **The default strategy should be "fine-tune," not "freeze."** The paper
  shows that a frozen transfer can perform needlessly poorly in the middle
  layers due to co-adaptation loss. Unless our datasets (mushroom, flower)
  are very small, fine-tuning the whole network at a low learning rate is
  the safer default.
- **Freezing early layers is low-risk; freezing late layers is high-risk.**
  A layer-wise `freeze_until_layer` parameter plus a `fine_tune_backbone:
  true/false` switch in the Model Zoo configs makes sense — this distinction
  comes directly from the paper's finding.
- **A randomly-initialized "no backbone" baseline is worth adding to the
  Benchmark sprint.** The paper shows this works fine for small networks in
  the literature but performs very poorly for deep networks — this would be
  a numerical control point supporting our decision to use an
  ImageNet-pretrained backbone.
- **The mushroom dataset carries an interesting detail:** in the paper's
  appendix list of man-made/natural classes, `mushroom` is directly listed
  under "natural." This suggests transfer is likely to work well due to
  conceptual proximity with ImageNet's mushroom classes (if any) — but
  ImageNet has only a limited number of mushroom classes, so this optimism
  shouldn't be overstated; it should be evaluated alongside the
  near-duplicate risk (consistent with the finding in our Sprint 1 notes).
- **The "transfer + fine-tuning beats training from scratch" finding**
  supports always including a "scratch" (randomly initialized) model
  variant in the Model Zoo's experiment matrix, and defaulting to a
  pretrained backbone even for the larger dataset.

**Open question (note for Sprint 5/6):** which layer counts as "middle"
depends on the backbone architecture used (the paper uses an AlexNet-like
8-layer network). If the Model Zoo uses much deeper architectures like
ResNet/EfficientNet, the "freeze the first N blocks" decision needs to be
recalibrated against block count — directly carrying over this paper's
layer numbers would not be correct.

## Reading Note 2: ImageNet: A Large-Scale Hierarchical Image Database

**Source:** Deng, J., Dong, W., Socher, R., Li, L.-J., Li, K., & Fei-Fei, L.
(2009). *ImageNet: A Large-Scale Hierarchical Image Database.* CVPR 2009.

### 1. What does the paper argue?

This is the paper behind the original ImageNet project — not the 1000-class
ILSVRC benchmark most people mean by "ImageNet" today, but the broader
effort: a visual ontology built on WordNet's noun hierarchy, aiming to
populate ~80,000 concepts (synsets) with 500–1,000 clean images each. The
2009 snapshot covers 12 subtrees (mammal, bird, vehicle, flower, ...), 5,247
synsets, and 3.2M images. Each synset sits in an IS-A hierarchy (e.g.
mammal → carnivore → canine → dog → husky), not a flat label list.

The paper argues ImageNet beats prior datasets (Caltech, TinyImage, ESP,
LabelMe) on four axes: label disambiguation (WordNet synsets resolve "bank"
river vs. bank building), label accuracy (~99.7%, human-verified), hierarchy
density, and full resolution (~400×350 vs. TinyImage's 32×32).

**Data collection (Section 3):** candidate images are pulled from multiple
search engines using WordNet synonyms (queries also translated into
Chinese/Spanish/Dutch/Italian for diversity), then filtered by Amazon
Mechanical Turk voters. The number of votes required per image isn't fixed —
easy concepts ("cat") need few votes, fine-grained ones ("Burmese cat") need
many independent votes — a dynamic confidence threshold that balances cost
against accuracy.

### Findings

- Clean, full-resolution data measurably beats noisy/low-res data on
  classification (tested with NN-voting / NBNN).
- A "tree-max classifier" — scoring a synset by the max of its own
  classifier and its children's — improves accuracy with no extra training.
  Leaf-near synsets (visually consistent, e.g. "star-nosed mole") classify
  more easily than root-near ones ("artifact", "vehicle").
- An early bounding-box localization experiment shows ImageNet can extend
  beyond class labels into spatial annotations.

## 3. Evaluation for VisionLab

This isn't a "model" paper, it's a "dataset philosophy" paper — no direct
architecture decision follows from it, but it backs two things we've
already built:

- **Label mapping ↔ synset IDs.** Our persistent JSON label mapping solves
  exactly the "label disambiguation" problem this paper describes — pinning
  a class name to one stable identity across the whole project, the same
  role ImageNet's WordNet synset IDs (`n02084071`-style) play instead of a
  human-readable name.
- **Near-duplicate risk ↔ intra-synset duplicate removal.** ImageNet treats
  duplicate filtering as a required step in its own collection pipeline —
  a reason to actually add a duplicate check (e.g. perceptual hashing) to
  the mushroom dataset, not just leave it as an open question.
- **Tree-max classifier is a cheap idea for later.** If the mushroom or
  flower dataset ever gets family/genus-level labels, scoring a class by
  the max across its hierarchy neighbors is a free accuracy gain worth
  trying in the Benchmark or Error Analysis sprint — no extra training
  needed.
- **Clean/full-resolution emphasis** supports keeping an image-quality
  check (corrupt files, very low resolution) in the shared
  `ImageClassificationDataset`, not just a broken-file check.

## Reading Note 3: Deep Residual Learning for Image Recognition

**Source:** He, K., Zhang, X., Ren, S., & Sun, J. (2016).
*Deep Residual Learning for Image Recognition.* CVPR 2016.
(ILSVRC 2015 classification winner)

### 1. What does the paper argue?

Starting question: does adding layers to a network always make it better?
No — and not because of vanishing/exploding gradients, which techniques
like batch normalization already keep in check. The authors call this the
**degradation problem**: past a certain depth, both training *and* test
error start rising — not overfitting, since training error itself gets
worse. A 56-layer "plain" network is empirically worse than an 18-layer one
on both train and test error (Fig. 1).

This is logically odd: a 56-layer network could always match an 18-layer
one by copying its layers and making the rest identity mappings — a
"no-worse-than" solution exists in theory, but SGD can't find it. The
authors' claim: learning identity mappings from scratch is hard for
*optimization*, not because the network lacks the capacity to represent
them.

**Solution — residual learning.** Instead of a block directly learning
`H(x)`, it learns the residual `F(x) := H(x) - x`, and outputs `F(x) + x`
(a "shortcut"/"skip" connection). If the optimal function really is close
to identity, pushing `F(x)` toward zero is a far easier optimization target
than reconstructing `H(x) = x` from scratch through nonlinear layers.
Shortcuts add no parameters (identity case) or negligible cost (1×1
projection case) and train end-to-end with plain SGD + backprop.

When channel counts don't match, either zero-padding (no extra params) or a
1×1 projection (few extra params) works — the gap between the two is small,
meaning the shortcut itself solves degradation; the projection choice is a
secondary detail.

**Bottleneck design:** for 50+ layer networks, a 3-layer bottleneck block
(1×1 reduce → 3×3 process → 1×1 restore) keeps compute in check — ResNet-152
ends up with *fewer* FLOPs than VGG-16/19 despite being far deeper.

### Findings

- 34-layer plain net is worse than 18-layer (degradation clearly visible);
  adding shortcuts (ResNet-34) beats the 18-layer ResNet — problem solved.
- Deeper consistently wins: ResNet-50 → 101 → 152 keep improving with no
  degradation (Table 3–4).
- Tested up to 1202 layers on CIFAR-10: 110-layer does well; 1202-layer
  reaches near-zero training error but slightly worse test error than the
  110-layer one — attributed to overfitting on a small dataset, not
  degradation.
- Layer-response std analysis (Fig. 7): ResNet layer outputs are generally
  smaller than plain nets, supporting the "residual functions stay close to
  zero" hypothesis.
- Generalizes beyond classification: swapping a VGG-16 backbone for
  ResNet-101 in COCO detection gives ~28% relative improvement with the
  same detection pipeline — attributed purely to representation quality.

## 4. Evaluation for VisionLab

This paper intersects directly with the Model Zoo sprint's architecture
choice and registry/factory pattern decisions.

**Concrete decisions this supports:**

- **A concrete reference point for backbone choice:** for fine-grained
  classification tasks like Mushroom and Flowers, this paper's evidence
  that performance improves systematically with depth strongly supports
  making a ResNet-50/101 backbone one of the Model Zoo's default/baseline
  options. Compared to VGG, it offers both fewer FLOPs and better
  accuracy — especially valuable if compute budget is limited.
- **The bottleneck / non-bottleneck distinction is a direct template for
  organizing registry model variants:** the architectural difference
  between ResNet-18/34 (plain 2-layer block) and ResNet-50/101/152
  (bottleneck block) can be modeled as a `block_type` parameter in our
  factory pattern — two different building-block classes sharing the same
  top-level network skeleton.
- **"Deeper is better, but only up to a point"** (the CIFAR-10 1202-layer
  experiment) is a warning for the Benchmark sprint: on small datasets (if
  our mushroom dataset ends up smaller than Flowers), an excessively deep
  backbone can raise overfitting risk. This makes it reasonable to tie
  model depth to dataset size via config (e.g. a shallower ResNet variant
  in `mushroom.yaml`, a deeper one in `flower.yaml`).
- **The "identity shortcut adds no parameters" principle** lines up with
  our checkpoint saving + metadata sidecar decision: when loading a ResNet
  checkpoint, recording which shortcut option (A/B/C) was used in the
  metadata may matter for correctly mapping pretrained weights coming from
  different sources (e.g. torchvision vs. timm).
- **Freezing BN statistics during fine-tuning** (Appendix A, detection
  section) is an interesting practical detail: when transfer learning, it's
  possible — and sometimes preferred, for memory/stability reasons — to
  freeze not just the weights but the BN layers' running mean/variance too.
  This adds a third option to the "freeze vs. fine-tune" discussion from
  our Yosinski note: the decision to freeze/fine-tune weights and
  normalization statistics separately. Worth considering as a config
  parameter (`freeze_bn_stats: true/false`) in the Training Engine sprint.
- **Direct link to our "`num_classes` is a mandatory parameter" decision:**
  the ResNet architecture is entirely generic up to the last layer (global
  average pooling + a single fully-connected layer) — that final layer is
  the only place tied to the number of classes. This architecturally
  validates our registry decision to require every model to take
  `num_classes` — the rest of the backbone is dataset-independent.

**Overall takeaway:** this paper answers a concrete "which architecture"
question for the Model Zoo sprint: the ResNet family (18/34/50/101/152) is
a depth/performance/cost spectrum that's easy to make parametric in a
registry. The bottleneck/non-bottleneck distinction and the BN-freezing
detail both feed directly into our factory pattern design and the Training
Engine's fine-tuning strategy.